In [10]:
import os
import json
import random

def calculate_penalty(proposed_group1, proposed_group2, history):
    penalty = 0
    for idx, past_meeting in enumerate(history):
        weight = 1.0 / (idx + 1) 
        past_groups = past_meeting['groups']
        past_g1, past_g2 = past_groups[0], past_groups[1]
        
        # --- 1. 사람 간의 중복 편성 벌점 ---
        for proposed_group in [proposed_group1, proposed_group2]:
            for i in range(len(proposed_group)):
                for j in range(i + 1, len(proposed_group)):
                    p1, p2 = proposed_group[i], proposed_group[j]
                    if (p1 in past_g1 and p2 in past_g1) or (p1 in past_g2 and p2 in past_g2):
                        penalty += weight
                        
        # --- 2. 특정 조 고착화 방지 벌점 ---
        for p in proposed_group1:
            if p in past_g1:
                penalty += weight * 0.5
                
        for p in proposed_group2:
            if p in past_g2:
                penalty += weight * 0.5
                
    return penalty

def divide_groups(attendees, meeting_date_str, folder_path="history"):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        
    history = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                history.append({
                    'date': filename.replace('.json', ''),
                    'groups': [data.get('group1', []), data.get('group2', [])]
                })
                
    history.sort(key=lambda x: x['date'], reverse=True)
    
    target_person = "오정환"
    has_target = target_person in attendees
    
    # [수정] 1조가 더 많아야 하므로 전체 인원 기준 1조 크기를 먼저 확정합니다.
    # 홀수일 때 n // 2 + 1이 되어 1조가 1명 더 많아집니다. (예: 7명일 때 size1 = 4)
    n = len(attendees)
    size1 = n // 2 + (1 if n % 2 != 0 else 0)
    
    pool = attendees.copy()
    if has_target:
        pool.remove(target_person)
    
    best_division = None
    min_penalty = float('inf')
    
    for _ in range(1000):
        random.shuffle(pool)
        
        if has_target:
            # [수정] 오정환은 무조건 2조로 가므로, 1조(g1)는 오정환을 뺀 풀에서 size1만큼 채웁니다.
            # 2조(g2)는 나머지 인원에 오정환을 합칩니다.
            g1 = pool[:size1]
            g2 = pool[size1:] + [target_person]
        else:
            # [수정] 오정환이 없을 때도 동일한 크기 기준으로 나눕니다.
            g1 = pool[:size1]
            g2 = pool[size1:]
        
        penalty = calculate_penalty(g1, g2, history)
        
        if best_division is None or penalty < min_penalty:
            min_penalty = penalty
            best_division = (list(g1), list(g2))
            
        if penalty == 0:
            break
            
    final_g1, final_g2 = best_division
    
    safe_date_str = meeting_date_str.replace('/', '-')
    filepath = os.path.join(folder_path, f"{safe_date_str}.json")
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump({"group1": final_g1, "group2": final_g2}, f, ensure_ascii=False, indent=2)
        
    return final_g1, final_g2

# --- 실행 부분 ---
# 총 7명 (홀수) 명단 구성
today_attendees = ["오정환", "김택민", "이동건", "박찬하", "현지용", "남궁시우", "한유안"]
today_date = "2026/06/28"

group1, group2 = divide_groups(today_attendees, today_date, './history')

print(f"[{today_date} 목장 모임 조 편성 결과]")
print(f"1조 ({len(group1)}명): {', '.join(group1)}")
print(f"2조 ({len(group2)}명): {', '.join(group2)}")

[2026/06/28 목장 모임 조 편성 결과]
1조 (4명): 현지용, 이동건, 김택민, 한유안
2조 (3명): 박찬하, 남궁시우, 오정환
